<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/Kalman/Example8wGOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

# --- CONFIGURATION & STRUCTURAL DEFINITIONS ---
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas'
]
n_states = len(state_elements)
n_periods = 6

# Mapping Matrix H_eia (Bridges the 8 hidden basin variables to 6 state EIA indicators)
H_eia = np.array([
    #P_O  EF_O H_O  Ot_O P_G  EF_G H_G  Ot_G
    [0.65, 1.0, 0.0, 0.70, 0.0, 0.0,  0.0,  0.0],  # TX Oil Total
    [0.35, 0.0, 0.0, 0.15, 0.0, 0.0,  0.0,  0.0],  # NM Oil Total
    [0.0,  0.0, 1.0, 0.15, 0.0, 0.0,  0.0,  0.0],  # LA Oil Total
    [0.0,  0.0, 0.0, 0.0,  0.65, 1.0, 0.10, 0.60], # TX Gas Total
    [0.0,  0.0, 0.0, 0.0,  0.35, 0.0, 0.0,  0.15], # NM Gas Total
    [0.0,  0.0, 0.0, 0.0,  0.0,  0.0, 0.90, 0.25]  # LA Gas Total
])

# --- DYNAMIC MATRIX PROFILE DICTIONARIES ---
# Historical Reporting Curves mapped by Age (Age 5 = Mature history, Age 0 = Bleeding Edge)
reporting_rates_by_age = {
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95]), # 5 Mos Ago (~95%)
    4: np.array([0.95, 0.94, 0.95, 0.93, 0.95, 0.94, 0.94, 0.93]),
    3: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]),
    2: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]),
    1: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]),
    0: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50])  # Current Mo (~50%)
}

# Standard deviations for Well-level data (Spikes heavily at Age 0)
well_std_by_age = {5: 15, 4: 25, 3: 50, 2: 120, 1: 250, 0: 500}
# Standard deviations for EIA top-down lines (Relatively stable across all months)
eia_std_by_age  = {5: 40, 4: 40, 3: 40, 2: 45,  1: 50,  0: 60}

# --- GENERATE TIME HORIZON ---
np.random.seed(42)
periods = pd.date_range(start="2026-03-01", periods=n_periods, freq="MS")
ages = [5, 4, 3, 2, 1, 0] # Matching timeline age pointers

# Core production values used as the true hidden baseline
true_base_values = np.array([3500, 1200, 50, 800, 2000, 3000, 14000, 4000])

fused_history = []

# --- STEP 1: LOOP THROUGH THE 6 PERIODS ---
for i, period in enumerate(periods):
    age = ages[i]
    rates = reporting_rates_by_age[age]

    # Generate Synthetic Actuals with a slight upward drift over the 6 months
    true_vals = true_base_values * (1.0 + 0.01 * i) + np.random.normal(0, 10, n_states)

    # 1. Gather Input Stream 1: Your Forecast (Stochastic prior variance)
    forecast_vals = true_vals + np.random.normal(0, 100, n_states)

    # 2. Gather Input Stream 2: Grossed-Up Well Data (Depressed by age-based reporting curve)
    well_raw = true_vals * rates + np.random.normal(0, 15, n_states)
    well_scaled = well_raw / rates

    # 3. Gather Input Stream 3: EIA-914 Top-Down State Totals
    eia_true_totals = np.dot(H_eia, true_vals)
    eia_observed = eia_true_totals + np.random.normal(0, 15, len(eia_true_totals))

    # --- STEP 2: MULTIVARIATE FILTER EXECUTION ---
    # Set initial state matrix using your forecast stream
    x = forecast_vals.copy()
    P = np.diag([120] * n_states)**2 # Prior forecast variance

    # UPDATE PHASE A: Fusing Well Data (H is Identity matrix)
    H_w = np.eye(n_states)
    R_w = np.diag([well_std_by_age[age]] * n_states)**2

    y_w = well_scaled - np.dot(H_w, x)
    S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
    K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
    x = x + np.dot(K_w, y_w)
    P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # UPDATE PHASE B: Fusing State EIA-914 Data
    R_e = np.diag([eia_std_by_age[age]] * len(eia_observed))**2

    y_e = eia_observed - np.dot(H_eia, x)
    S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
    K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
    x = x + np.dot(K_e, y_e)

    # Store complete dataset for evaluation
    fused_history.append({
        'Period': period.strftime('%Y-%m'),
        'Age': age,
        'Permian_Oil_Forecast': forecast_vals[0],
        'Permian_Oil_Wells_Scaled': well_scaled[0],
        'Permian_Oil_Fused': x[0],
        'Permian_Oil_True': true_vals[0],
        'Haynesville_Gas_Forecast': forecast_vals[6],
        'Haynesville_Gas_Wells_Scaled': well_scaled[6],
        'Haynesville_Gas_Fused': x[6],
        'Haynesville_Gas_True': true_vals[6],
    })

# --- STEP 3: ANALYZE ERROR PROFILE ---
df_results = pd.DataFrame(fused_history)
print("=== MULTI-PERIOD DATA FUSION OVERVIEW ===")
print(df_results.round(0).to_string(index=False))


=== MULTI-PERIOD DATA FUSION OVERVIEW ===
 Period  Age  Permian_Oil_Forecast  Permian_Oil_Wells_Scaled  Permian_Oil_Fused  Permian_Oil_True  Haynesville_Gas_Forecast  Haynesville_Gas_Wells_Scaled  Haynesville_Gas_Fused  Haynesville_Gas_True
2026-03    5                3458.0                    3489.0             3490.0            3505.0                   13843.0                       14017.0                14015.0               14016.0
2026-04    4                3396.0                    3522.0             3522.0            3529.0                   13994.0                       14131.0                14133.0               14142.0
2026-05    3                3601.0                    3567.0             3569.0            3565.0                   14017.0                       14304.0                14275.0               14279.0
2026-06    2                3606.0                    3639.0             3624.0            3606.0                   14407.0                       14424.0         

The Stability Shift (Age 5 to 3): For early months like 2026-03, the well data variance (well_std_by_age) is incredibly small (15). The filter relies directly on the Permian_Oil_Wells_Scaled lines, quickly overriding structural anomalies in your base forecast.The

Bleeding-Edge Protection (Age 0): For the final period (2026-08), the raw well counts provide minimal visibility. The model identifies that well_std_by_age has surged to 500, meaning it essentially bypasses the unstable scaled well volume, anchors tightly to the EIA state total balance constraints via the mapping matrix, and blends them with your forecast to land right next to the true production line.

K_e is exactly the Kalman Gain [1] for the second phase of the data fusion process (updating the model with the top-down State EIA-914 data).In a standard sequential Kalman Filter, you compute a unique Kalman Gain vector or matrix for every distinct observation stream you feed into the model.

In the 6-period code example:
K_w is the Kalman Gain calculated for your bottom-up well-level data.
K_e is the Kalman Gain calculated for your top-down state EIA data.

How K_e Acts as an Automated Weighting ValveMathematically, the Kalman Gain acts as a weighting mechanism that evaluates the relative certainty between your model's current belief and the incoming observation.Look at how K_e changes its behavior dynamically depending on the reporting maturity (Age) of the data:At Age 0 (50% reported well data): The well data is highly uncertain (R_w is massive). Consequently, the first gain factor (K_w) drops close to zero, and the model leaves the state estimate largely unchanged. When the loop hits the EIA data phase, the system realizes the EIA line is far more reliable than the messy well records (R_e is much smaller than R_w). K_e automatically scales upward, forcing the final estimate to anchor tightly to the EIA state balance constraints.At Age 5 (95%+ reported well data): The well records are now clear, solid, and reliable (R_w is tiny). The first gain factor (K_w) scales way up, instantly pinning the state estimate to the well data. By the time the code calculates K_e, the model's updated internal variance matrix (P) has shrunk to near-zero because it already found a highly certain data match. K_e automatically dials down to near-zero, preventing the less granular EIA figures from overriding or distorting the clean well data.

 Why GOR Refines the Estimates (The Core Advantage)In oil and gas data streams, gas data and oil data do not drop at the same rate, but their physical relationship is bound by the reservoir.In the ~50% Month (Age 0): A state registry might accidentally process a massive batch of natural gas volume reports for a group of new Permian wells while their corresponding crude oil volumes are stuck in a regulatory lease-accounting backlog.Without GOR: The current model sees a massive spike in partial gas data and a drop in oil data. It might overcorrect by scaling up your gas estimate while leaving your oil estimate depressed.With GOR: The model uses the physical GOR link to say: "Wait, if gas production is actually this high, the physical GOR dictatess that oil production must be higher too, despite what the partial well files say." It will automatically lift the oil estimate to match.

In [2]:
import numpy as np
import pandas as pd

# --- STRUCTURAL CONFIGURATION ---
# The state vector now tracks 12 variables: 4 Oil Basins, 4 Gas Basins, and 4 Basin GORs
state_elements = [
    'Permian_Oil', 'EagleFord_Oil', 'Haynesville_Oil', 'Other_Oil',
    'Permian_Gas', 'EagleFord_Gas', 'Haynesville_Gas', 'Other_Gas',
    'Permian_GOR', 'EagleFord_GOR', 'Haynesville_GOR', 'Other_GOR'
]
n_states = len(state_elements)
n_periods = 6

# Mapping Matrix H_eia (Bridges the 12 hidden variables to the 6 state EIA indicators)
# Columns 8-11 (the GOR states) are 0 because EIA does not report a standalone GOR number.
H_eia = np.zeros((6, n_states))
H_eia[0, 0:4] = [0.65, 1.0, 0.0, 0.70] # TX Oil Total
H_eia[1, 0:4] = [0.35, 0.0, 0.0, 0.15] # NM Oil Total
H_eia[2, 0:4] = [0.0,  0.0, 1.0, 0.15] # LA Oil Total
H_eia[3, 4:8] = [0.65, 1.0, 0.10, 0.60] # TX Gas Total
H_eia[4, 4:8] = [0.35, 0.0, 0.0,  0.15] # NM Gas Total
H_eia[5, 4:8] = [0.0,  0.0, 0.90, 0.25] # LA Gas Total

# --- VARIANCE PROFILES BY MONTH AGE ---
reporting_rates_by_age = {
    5: np.array([0.96, 0.95, 0.97, 0.95, 0.96, 0.95, 0.95, 0.95]), # 5 Mos Ago (~95%)
    4: np.array([0.95, 0.94, 0.95, 0.93, 0.95, 0.94, 0.94, 0.93]),
    3: np.array([0.90, 0.88, 0.92, 0.87, 0.91, 0.89, 0.88, 0.86]),
    2: np.array([0.85, 0.80, 0.86, 0.82, 0.84, 0.81, 0.78, 0.80]),
    1: np.array([0.70, 0.65, 0.72, 0.68, 0.71, 0.66, 0.62, 0.64]),
    0: np.array([0.52, 0.48, 0.60, 0.50, 0.55, 0.50, 0.45, 0.50])  # Current Mo (~50%)
}

well_std_by_age = {5: 15, 4: 25, 3: 50, 2: 120, 1: 250, 0: 500}
eia_std_by_age  = {5: 40, 4: 40, 3: 40, 2: 45,  1: 50,  0: 60}

# Enforce tight GOR linkage at Age 0 (std=0.1), let it breathe at Age 5 (std=50.0)
# gor_constraint_std_by_age = {5: 50.0, 4: 20.0, 3: 5.0, 2: 1.0, 1: 0.2, 0: 0.1}

# NEW LOOSENED GOR PROFILE:
gor_constraint_std_by_age = {
    5: 1_000_000.0,  # 5 Mos Ago (~95%+): Effectively infinity. Completely ignores GOR.
    4: 1_000_000.0,  # 4 Mos Ago (~95%+): Completely ignores GOR.
    3: 50_000.0,     # 3 Mos Ago (~90%+): Very loose. Lets the mature actual data dominate.
    2: 1.0,          # 2 Mos Ago (~85%): Moderate constraint.
    1: 0.2,          # 1 Mo Ago (~70%): Tight constraint.
    0: 0.1           # Current Month (~50%): Ultra-tight constraint. GOR rescues missing data.
}








# --- GENERATE DATA HORIZON ---
np.random.seed(42)
periods = pd.date_range(start="2026-03-01", periods=n_periods, freq="MS")
ages = [5, 4, 3, 2, 1, 0]

# True underlying values (Oil base, Gas base, GOR base)
true_oil_base = np.array([3500, 1200, 50, 800])
true_gor_base = np.array([2.5, 3.0, 150.0, 4.0])
true_gas_base = true_oil_base * true_gor_base

fused_history = []

for i, period in enumerate(periods):
    age = ages[i]
    rates = reporting_rates_by_age[age]

    # 1. Generate True Values for this month (with dynamic monthly variance)
    true_oil = true_oil_base * (1.0 + 0.01 * i) + np.random.normal(0, 10, 4)
    true_gor = true_gor_base + np.random.normal(0, 0.05, 4) # GOR naturally drifts slightly
    true_gas = true_oil * true_gor
    true_vector = np.concatenate([true_oil, true_gas, true_gor])

    # 2. Gather Input Stream 1: Prior Forecast
    forecast_vector = true_vector + np.random.normal(0, 100, n_states)
    forecast_vector[8:12] = true_gor + np.random.normal(0, 0.2, 4) # Forecast GOR accuracy

    # 3. Gather Input Stream 2: Grossed-Up Well Data (only targets first 8 volume elements)
    well_volumes_true = true_vector[0:8]
    well_raw = well_volumes_true * rates + np.random.normal(0, 15, 8)
    well_scaled = well_raw / rates

    # 4. Gather Input Stream 3: EIA-914 Top-Down State Totals
    eia_observed = np.dot(H_eia, true_vector) + np.random.normal(0, 15, 6)

    # --- STEP 2: KALMAN FILTER DATA FUSION WITH EXTENDED GOR LAYER ---
    x = forecast_vector.copy()
    P = np.eye(n_states) * 100**2
    P[8:12, 8:12] = np.eye(4) * 0.5**2 # Variance of GOR forecast belief

    # PHASE A: Update using Partial Basin Well-Level Data (8 volume inputs)
    H_w = np.zeros((8, n_states))
    H_w[0:8, 0:8] = np.eye(8)
    R_w = np.eye(8) * (well_std_by_age[age]**2)

    y_w = well_scaled - np.dot(H_w, x)
    S_w = np.dot(H_w, np.dot(P, H_w.T)) + R_w
    K_w = np.dot(P, np.dot(H_w.T, np.linalg.inv(S_w)))
    x = x + np.dot(K_w, y_w)
    P = np.dot(np.eye(n_states) - np.dot(K_w, H_w), P)

    # PHASE B: Update using State EIA-914 Data (6 inputs)
    R_e = np.eye(6) * (eia_std_by_age[age]**2)

    y_e = eia_observed - np.dot(H_eia, x)
    S_e = np.dot(H_eia, np.dot(P, H_eia.T)) + R_e
    K_e = np.dot(P, np.dot(H_eia.T, np.linalg.inv(S_e)))
    x = x + np.dot(K_e, y_e)
    P = np.dot(np.eye(n_states) - np.dot(K_e, H_eia), P)

    # PHASE C: The Fully Integrated GOR Matrix Update (4 constraints)
    # Re-linearize the physical equation around current estimates: Gas - (GOR * Oil) = 0
    H_gor = np.zeros((4, n_states))
    for b in range(4):
        current_oil = x[b]
        current_gor = x[b+8]
        H_gor[b, b]   = -current_gor # Derivative with respect to Oil
        H_gor[b, b+4] = 1.0          # Derivative with respect to Gas
        H_gor[b, b+8] = -current_oil # Derivative with respect to GOR

    # The mathematical target for the residual equation is always 0
    gor_residuals = np.zeros(4) - (x[4:8] - (x[0:4] * x[8:12]))

    # Assign the dynamic age-dependent constraint variance
    R_gor = np.eye(4) * (gor_constraint_std_by_age[age]**2)

    S_gor = np.dot(H_gor, np.dot(P, H_gor.T)) + R_gor
    K_gor = np.dot(P, np.dot(H_gor.T, np.linalg.inv(S_gor)))
    x = x + np.dot(K_gor, gor_residuals)
    P = np.dot(np.eye(n_states) - np.dot(K_gor, H_gor), P)

    fused_history.append({
        'Period': period.strftime('%Y-%m'),
        'Age': age,
        'Permian_Oil_Forecast': forecast_vector[0],
        'Permian_Oil_Wells_Scaled': well_scaled[0],
        'Permian_Oil_Fused': x[0],
        'Permian_Oil_True': true_vector[0],
        'Permian_Gas_Forecast': forecast_vector[4],
        'Permian_Gas_Wells_Scaled': well_scaled[4],
        'Permian_Gas_Fused': x[4],
        'Permian_Gas_True': true_vector[4],
        'Permian_GOR_Fused': x[8],
        'Permian_GOR_True': true_vector[8]
    })

df_results = pd.DataFrame(fused_history)
print("=== GOR-INTEGRATED MULTI-PERIOD ESTIMATION ===")
print(df_results.round(1).to_string(index=False))


=== GOR-INTEGRATED MULTI-PERIOD ESTIMATION ===
 Period  Age  Permian_Oil_Forecast  Permian_Oil_Wells_Scaled  Permian_Oil_Fused  Permian_Oil_True  Permian_Gas_Forecast  Permian_Gas_Wells_Scaled  Permian_Gas_Fused  Permian_Gas_True  Permian_GOR_Fused  Permian_GOR_True
2026-03    5                3458.0                    3496.5             3494.7            3505.0                8745.6                    8712.0             8711.6            8721.4                2.5               2.5
2026-04    4                3475.7                    3504.2             3508.7            3521.7                8816.3                    8782.8             8782.4            8783.9                2.5               2.5
2026-05    3                3490.0                    3547.3             3549.3            3570.9                8835.0                    8885.3             8870.5            8888.0                2.5               2.5
2026-06    2                3743.4                    3617.4             

Why the results are much better:Phase C is an Extended Kalman Filter (EKF) step: Because \(Gas = Oil \times GOR\) is a non-linear relationship, the loop recalculates the derivatives (H_gor) using the updated variables in real time.The Cross-Pollination Benefit: At Age 0, if your Permian_Oil_Wells_Scaled number encounters a bad reporting slump, the model relies on Phase C to look at the gas stream and the tracked GOR state to immediately force the oil estimate to correct itself.

https://www.google.com/search?q=in+some+series+we+get+delay+reporting.+but+some+data+is+already+suggesting+the+outcome.+how+to+predict+the+outcome+with+partial+data%3F&ie=UTF-8&oe=UTF-8&hl=en-us&client=safari&fbs=ABfTbFUxGEP8yeZbmk97ajdTjIq-Fell6yjIojusYtuKjXhLi43HlmHdBhkjA3l1LeWMNI31J3wjh5ota8viteVD7M7JWhWI1sJN_Y-p91HosJvxELbf3xOAD9pl09itgH0OE_NT6Mg1wyQNC-omONpSlPhrmGWkLje8lo-5dZEjjSqkJ51QKkUlCt8SiG48ELLsMYZnaxdIrMwCYEaqwpjhOooydHJfwOCThv4Z87TBXWVMdIg1kwg&aep=10&ntc=1&sxsrf=APpeQnudhjQ4f-WzAmQ4qemAEy0xtX6OFA:1788520140838&mstk=AUtExfDhAGI7_eoTJoQdnkqXCk1dNcX0Iufjkynm3VMli7iyaHYeV2fcCfMs_OT1TZUs8uDn2e1mukzCbiBz6wSxfK1CZp0Dkch_ch_lckW0lcQoS4a_T9TL1IfxMHWMvLvU3_gZNfrnQR_wD6qSfs-Cp3lQbm8K0k3O35AsT3aIjd4W2DcJnuwKCkX6u73pSrRIAL56woHhUldGmq0CiJfITSKHmBDIN4xBlvn5LtNXz2wPsKTmYIRC1ceLx37RhoYaIIpTMpaw8zjzYZ1ynKqWBsolM_GAnDzuijhGGaYw7z0UyAfWxYjuzwjPRHEtgMvHsET-cJ6f8ZMFYsttujNBbz4BMrZD3kxi3Q&csuir=1&aioh=3&atvm=2&mtid=QaiaatT8BuznwbkPquf_uA8&udm=50&iga=1&utm_campaign=safari_share_1